# Demo — pipeline multi-tenant SAAS

Notebook para la sustentación. Usa el paquete `saas_pipeline` (no duplica lógica): ejecuta el
pipeline y muestra sus resultados.

Se abre con `make notebook` (Jupyter del entorno del proyecto, en WSL) y se ejecuta de arriba a
abajo. Cambiar `TENANT` para mostrar otro país.

In [ ]:
import logging
import os
from pathlib import Path

# Run from the repository root so config/ and data/raw/ resolve as in the CLI.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

from delta.tables import DeltaTable
from pyspark.sql import functions as F

from saas_pipeline import bronze, gold, silver
from saas_pipeline.config import load_config, quarantine_path, table_path
from saas_pipeline.pipeline import LAYER_ORDER, run
from saas_pipeline.session import get_spark

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
spark = get_spark("demo")
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)  # DataFrames render as HTML tables
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 30)

## 1. Parámetros

In [ ]:
ENV = "dev"
TENANT = "sv"
START, END = "2025-01-01", "2025-06-30"

cfg = load_config(ENV, TENANT, [f"execution.start_date={START}", f"execution.end_date={END}"])


def load(path: str):
    """Read a Delta table by path."""
    return spark.read.format("delta").load(path)


def latest_run_checks(tenant: str):
    """Quality checks of the most recent run of a tenant."""
    logs = load(cfg.paths.quality_logs).filter(F.col("tenant_id") == tenant)
    last_run = logs.orderBy(F.desc("executed_at")).first()["_run_id"]
    return logs.filter(F.col("_run_id") == last_run).select(
        "layer", "check_name", "check_severity", "records_checked", "records_failed", "check_passed"
    )


print("data root:", cfg.paths.root)

## 2. Ejecutar el pipeline para un tenant

Bronze → Silver → checks de calidad → Gold. Los logs muestran la conciliación de Silver:
leídas = duplicados + descartadas + cuarentena + escritas.

In [ ]:
exit_code = run(
    ENV, TENANT, list(LAYER_ORDER), [f"execution.start_date={START}", f"execution.end_date={END}"]
)
print("exit code:", exit_code)

## 3. Calidad: checks de la última corrida (`quality_logs`)

In [ ]:
latest_run_checks(TENANT)

## 4. Gold: `daily_metrics_by_delivery_type`

Revenue = cantidad normalizada a ST × precio de la transacción.

In [ ]:
load(table_path(cfg, "gold", gold.TABLE)).orderBy("fecha_proceso", "tipo_entrega").drop(
    "_batch_id", "_processed_at"
)

## 4b. Gold: `top_materials_by_month` (bonus)

Top materiales por revenue, por mes. Cada mes se recalcula completo aunque el rango procesado
sea parcial; `categoria` es la última versión del material usada en el mes (SCD2).

In [ ]:
load(table_path(cfg, "gold", gold.TOP_TABLE)).orderBy("month_start", "revenue_rank").drop(
    "_batch_id", "_processed_at"
)

## 5. Cuarentena por motivo (Bronze y Silver)

In [ ]:
bronze_q = load(quarantine_path(cfg, "bronze", bronze.TABLE)).groupBy("_quarantine_reason").count()
silver_q = (
    load(quarantine_path(cfg, "silver", silver.FACT_TABLE)).groupBy("_quarantine_reason").count()
)
bronze_q.withColumn("layer", F.lit("bronze")).unionByName(
    silver_q.withColumn("layer", F.lit("silver"))
).orderBy("layer", F.desc("count"))

## 6. SCD Type 2: join temporal, no `is_current`

DA030001 cambia de precio el 2025-02-01 (55 → 57,5) y el 2025-05-16 (57,5 → 60). Cada entrega
toma la versión vigente en su fecha; con `is_current` todas valdrían 60.

In [ ]:
fact = load(table_path(cfg, "silver", silver.FACT_TABLE))
(
    fact.filter(F.col("material").isin("DA030001", "BA018426"))
    .groupBy("material", F.substring("fecha_proceso", 1, 6).alias("mes"))
    .agg(
        F.collect_set("precio_base").alias("precio_base"),
        F.collect_set("categoria").alias("categoria"),
    )
    .orderBy("material", "mes")
)

## 7. Idempotencia: historial de Delta

Volver a ejecutar el mismo rango no duplica: Bronze reemplaza el rango (`WRITE` con
`replaceWhere`) y Silver hace `MERGE` (actualiza las filas existentes, no inserta nuevas).

In [ ]:
before = {"bronze": load(table_path(cfg, "bronze", bronze.TABLE)).count(), "silver": fact.count()}
run(ENV, TENANT, list(LAYER_ORDER), [f"execution.start_date={START}", f"execution.end_date={END}"])
after = {
    "bronze": load(table_path(cfg, "bronze", bronze.TABLE)).count(),
    "silver": load(table_path(cfg, "silver", silver.FACT_TABLE)).count(),
}
print("antes:", before, "| después:", after, "| iguales:", before == after)

In [ ]:
(
    DeltaTable.forPath(spark, table_path(cfg, "silver", silver.FACT_TABLE))
    .history(4)
    .select(
        "version",
        "operation",
        F.col("operationMetrics.numTargetRowsInserted").alias("inserted"),
        F.col("operationMetrics.numTargetRowsUpdated").alias("updated"),
        F.col("operationMetrics.numTargetRowsDeleted").alias("deleted"),
    )
)

## 8. Reproceso de un rango (solo marzo)

Solo cambian de lote (`_batch_id`) las filas de marzo; el resto del semestre no se toca.

In [ ]:
run(
    ENV,
    TENANT,
    list(LAYER_ORDER),
    ["execution.start_date=2025-03-01", "execution.end_date=2025-03-31"],
)
(
    load(table_path(cfg, "bronze", bronze.TABLE))
    .groupBy(F.substring("fecha_proceso", 1, 6).alias("mes"))
    .agg(
        F.countDistinct("_batch_id").alias("lotes"),
        F.max("_batch_id").alias("ultimo_lote"),
        F.count("*").alias("filas"),
    )
    .orderBy("mes")
)

## 9. Consultas libres

Vistas temporales del tenant para responder preguntas en vivo con SQL.

In [ ]:
views = {
    "bronze_deliveries": table_path(cfg, "bronze", bronze.TABLE),
    "fact_deliveries": table_path(cfg, "silver", silver.FACT_TABLE),
    "dim_materials": table_path(cfg, "silver", silver.DIM_TABLE),
    "silver_quarantine": quarantine_path(cfg, "silver", silver.FACT_TABLE),
    "gold_daily_metrics": table_path(cfg, "gold", gold.TABLE),
    "quality_logs": cfg.paths.quality_logs,
}
for name, path in views.items():
    load(path).createOrReplaceTempView(name)

spark.sql("""
    SELECT tipo_entrega, ROUND(SUM(total_revenue), 2) AS revenue, SUM(total_units) AS units
    FROM gold_daily_metrics
    GROUP BY tipo_entrega
    ORDER BY revenue DESC
""")